# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [3]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5051 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [9]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for a in args:
            if isinstance(a, (list)):
                print(len(a))
        for v in kwargs.values():
            if isinstance(v, (list)):
                print(len(v))

        return func(*args, **kwargs)
    return wrapper

# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie: {name}")
    print("---------------------------")

process_data(["mleko", "chleb", "masło", "jajka"], "Lista zakupów Kacpra")

process_data(name="Franek", data_list=[1, 2, 3, 4, 5, 6])


4
Przetwarzanie: Lista zakupów Kacpra
---------------------------
6
Przetwarzanie: Franek
---------------------------


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [ ]:
import functools
from datetime import datetime
import time

def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time.time()
            current_date = datetime.now()
            result = func(*args, **kwargs)  
            end_time = time.time()
            execute_time = end_time - start_time

            with open(filename, "a", encoding="utf-8") as file:
                file.write(f"Function: {func.__name__} | Date: {current_date} | Execution time: {execute_time} s\n") 
            return result
        return wrapper
    return decorator

@logger("execution_history.log")
def huge_amount(n):
    print(f"Started counting to {n}...")
    sum = 0
    for i in range(n):
        sum += i
    print("Finished counting")
    return sum

result = huge_amount(5000000)
print(f"Wynik dodawania: {result}")

Started counting to 5000000...
Finished counting
Wynik dodawania: 12499997500000


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [6]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [17]:
class AccessLogger:
    def __init__(self, name):
        self.name = name
    
    def __get__(self, instance, owner):
        print(f"Getting atributes: {self.name}")
        return instance.__dict__.get(self.name)
    
    def __set__(self, instance, value):
        print(f"Settings atributes: {self.name} has value {value}")
        instance.__dict__[self.name] = value


class Uzytkownik:
    name = AccessLogger("name")
    age = AccessLogger("age")


print("============= Create user =============")
user_1 = Uzytkownik()

print("\n=============  Set value ============= ")
user_1.name = "Kacper"
user_1.age = 22

print("\n=============  Get value ============= ")
print(f"Name is: {user_1.name}")
print(f"Age is: {user_1.age}")


============= Create user =============

=============  Set value ============= 
Settings atributes: name has value Kacper
Settings atributes: age has value 22

=============  Get value ============= 
Getting atributes: name
Name is: Kacper
Getting atributes: age
Age is: 22


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [3]:
def collatz_generator(n):
    yield n 
    while n > 1:
        if n % 2 == 0:
            n = n // 2
        else:
            n = n * 3 + 1 
        yield n


# Test:
for status in collatz_generator(10):
    print(status)

10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [8]:
import functools

current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if role != current_user["role"]:
                raise PermissionError("Unauthorized user. Cannot proceed the action")
            else:
                print("You have access!")
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@require_role("superuser")
def delete_database():
    print("Database has been deleted.")

@require_role("user")
def add_comment():
    print("Comment has been added.")

delete_database()
add_comment()


You have access!
Database has been deleted.


PermissionError: Unauthorized user. Cannot proceed the action

### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [ ]:
class Typed:
    def __init__(self, name, expected_type):
        self.name = name
        self.expected_type = expected_type
    def __get__(self, instance, owner):
        return instance.__dict__.get(self.name)
    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError("Incrrect value type")
        else:
            instance.__dict__[self.name] = value

class Osoba():
    name = Typed("name", str)
    age = Typed("age", int)

user = Osoba()
user.name = "Kacper"
user.age = 22

print(f"Użytkownik: {user.name}, Wiek: {user.age}")

Użytkownik: Kacper, Wiek: None


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [ ]:
# TODO: Implementacja generatora liczb pierwszych
def prime_generator():
    pass